# 04 - Train XGBoost Models

Train the 20 XGBoost regressors with 144 combinations used in the dissertation methodology, then review the saved per-target metrics.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

project_root = '/content/drive/MyDrive/Dissertation_ML_Project_V1'
if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)
print(f'Project root set to: {project_root}')


In [ ]:
!pip install xgboost -q

from src.train_xgb import train_all_xgb

train_all_xgb("data/processed", "models/XGBoost", "tables")

In [ ]:
import json
import pandas as pd

with open('models/XGBoost/xgb_results.json') as f:
    results = json.load(f)

rows = []
for target, result in results.items():
    if 'error' in result:
        rows.append({'target': target, 'status': 'ERROR'})
        continue
    rows.append({
        'target': target,
        'r2_cv': result.get('r2_cv'),
        'r2_test': result.get('r2_test'),
        'r2_train': result.get('r2_train'),
        'rmse': result.get('rmse'),
        'mae': result.get('mae'),
        'MU': result.get('max_underestimate'),
        'overfit_flag': result.get('overfit_flag', False),
        'status': result.get('status'),
        'time_min': result.get('train_time_seconds', 0) / 60,
    })

df = pd.DataFrame(rows)
print(f'Total models: {len(df)}')
print(f'R² >= 0.96: {(df["r2_test"] >= 0.96).sum()}/20')
print(f'Overfitting (train-test > 0.03): {df["overfit_flag"].sum()}/20')
df
